# v46 — Euclid P3: Topological Node Density
**Status April 2026:** Notebook bereit. Warte auf Euclid DR1 (21. Oktober 2026).
Synthetische Daten als Platzhalter. Bei Release: FILAMENT_FILE und GALAXY_FILE ersetzen.

**Pre-registriert:** März 2026. Vorhersagen unveränderlich.

**Alle wissenschaftlichen Ideen: Andrew Cotting. KI-Assistenz deklariert (Claude, Anthropic).**

In [ ]:
# v46 -- Euclid P3: Topological Node Density Test
# Pre-registered prediction:
#   ~13 galaxy-density nodes in Euclid volume
#   Node spacing: 1315 x 1375 Mpc (y x z)
#   Nodes concentrated near x=0 identification plane
#
# KTF3: Klein-bottle identification phi(x,y,z) = (-x,y,z)
# fixes x=0 plane. Nodes = intersections with period boundaries:
#   y = n * T2/2 = n * 1315 Mpc
#   z = m * T3/2 = m * 1375 Mpc
#
# PRE-REGISTERED: March 2026, before Euclid DR1 release.
# Cotting, S. (2026)

In [ ]:
# CELL 1 -- Imports
!pip install numpy matplotlib scipy astropy -q
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
from scipy.signal import find_peaks
import os, warnings
warnings.filterwarnings('ignore')

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

# Topological parameters
T1, T2, T3 = 1660, 2630, 2750  # Mpc
node_dy = T2 / 2  # = 1315 Mpc
node_dz = T3 / 2  # = 1375 Mpc

print('v46 -- Euclid P3: Topological Node Density')
print('PRE-REGISTERED March 2026 -- Cotting, S.')
print('='*60)
print()
print('KTF3 Prediction P3:')
print(f'  Node spacing y: T2/2 = {node_dy} Mpc')
print(f'  Node spacing z: T3/2 = {node_dz} Mpc')
print(f'  Expected nodes in Euclid: ~13')
print(f'  Location: near x=0 identification plane')
print()
print('Falsified if: node positions consistent with random LCDM')

In [ ]:
# CELL 2 -- Load Euclid data
FILAMENT_FILE = 'euclid_dr1_filaments.fits'  # << REPLACE WITH ACTUAL FILE
GALAXY_FILE   = 'euclid_dr1_galaxies.fits'   # << REPLACE WITH ACTUAL FILE

if not os.path.exists(FILAMENT_FILE):
    print('Euclid data not yet available. Generating synthetic data...')
    N_nodes_synthetic = 15
    rng = np.random.default_rng(42)
    
    # Synthetic filament intersection nodes
    # Random positions + a few at predicted locations
    N_random = 200
    chi_max  = 5312  # Mpc (Euclid depth)
    
    node_x = rng.uniform(-chi_max/2, chi_max/2, N_random)
    node_y = rng.uniform(-chi_max/2, chi_max/2, N_random)
    node_z = rng.uniform(-chi_max/2, chi_max/2, N_random)
    
    # Add synthetic signal nodes near x=0, at predicted y,z spacings
    for ny in range(-3, 4):
        for nz in range(-3, 4):
            node_x = np.append(node_x, rng.normal(0, 100))
            node_y = np.append(node_y, ny * node_dy + rng.normal(0, 50))
            node_z = np.append(node_z, nz * node_dz + rng.normal(0, 50))
    
    nodes = np.stack([node_x, node_y, node_z], axis=1)
    USE_SYNTHETIC = True
    print(f'Synthetic nodes: {len(nodes)} (including {7*7} signal nodes)')
else:
    # Load real Euclid filament nodes
    # Nodes = filament intersection points from DisPerSE/T-ReX
    hdul = fits.open(FILAMENT_FILE)
    data = hdul[1].data
    # Extract node positions (format depends on DisPerSE output)
    ra_n  = data['RA_NODE']  if 'RA_NODE'  in data.dtype.names else data['RA']
    dec_n = data['DEC_NODE'] if 'DEC_NODE' in data.dtype.names else data['DEC']
    z_n   = data['Z_NODE']   if 'Z_NODE'   in data.dtype.names else data['Z']
    chi_n = cosmo.comoving_distance(z_n).value
    r_n, d_n = np.radians(ra_n), np.radians(dec_n)
    node_x = chi_n * np.cos(d_n) * np.cos(r_n)
    node_y = chi_n * np.cos(d_n) * np.sin(r_n)
    node_z = chi_n * np.sin(d_n)
    nodes  = np.stack([node_x, node_y, node_z], axis=1)
    USE_SYNTHETIC = False
    print(f'Nodes loaded: {len(nodes)}')

In [ ]:
# CELL 3 -- Test P3: Node positions vs predicted grid
# Test 1: Are nodes concentrated near x=0 plane?
# Test 2: Do nodes show periodic spacing in y and z at T2/2 and T3/2?

print('Test 1: Node concentration near x=0 plane')
x_width = 500  # Mpc -- width of x=0 plane region
n_near_x0  = (np.abs(nodes[:,0]) < x_width).sum()
n_total    = len(nodes)
frac_x0    = n_near_x0 / n_total
frac_expect = 2*x_width / (nodes[:,0].max() - nodes[:,0].min())
enhancement = frac_x0 / frac_expect
print(f'  Nodes near |x| < {x_width} Mpc: {n_near_x0}/{n_total} = {frac_x0:.3f}')
print(f'  Expected (random): {frac_expect:.3f}')
print(f'  Enhancement factor: {enhancement:.2f}x')
print()

print('Test 2: Periodic spacing in y-direction')
# Compute pairwise y-separations
dy_pairs = []
for i in range(min(len(nodes), 500)):
    for j in range(i+1, min(len(nodes), 500)):
        dy_pairs.append(abs(nodes[i,1] - nodes[j,1]))
dy_pairs = np.array(dy_pairs)

# Histogram of y-separations
dy_bins = np.arange(0, 4000, 100)
dy_hist, _ = np.histogram(dy_pairs, bins=dy_bins)
dy_centers = 0.5*(dy_bins[:-1]+dy_bins[1:])

# Look for peak at node_dy = 1315 Mpc
pred_bin_y = np.argmin(np.abs(dy_centers - node_dy))
pred_bin_z = np.argmin(np.abs(dy_centers - node_dz))
mean_count = dy_hist.mean()
excess_y   = dy_hist[pred_bin_y] / mean_count - 1
excess_z   = dy_hist[pred_bin_z] / mean_count - 1

print(f'  Peak at predicted y-spacing {node_dy} Mpc: excess = {excess_y:.3f}')
print(f'  Peak at predicted z-spacing {node_dz} Mpc: excess = {excess_z:.3f}')
print(f'  (Positive = more pairs than average at predicted spacing)')

In [ ]:
# CELL 4 -- Monte Carlo significance for node spacing
N_MC = 200
print(f'Monte Carlo null (N={N_MC}): randomising node positions...')
rng_mc = np.random.default_rng(99)
mc_excess_y = []
mc_excess_z = []

for i in range(N_MC):
    # Randomise node y positions within survey volume
    y_rand = rng_mc.uniform(nodes[:,1].min(), nodes[:,1].max(), len(nodes))
    z_rand = rng_mc.uniform(nodes[:,2].min(), nodes[:,2].max(), len(nodes))
    
    dy_rand = []
    for j in range(min(len(nodes), 500)):
        for k in range(j+1, min(len(nodes), 500)):
            dy_rand.append(abs(y_rand[j] - y_rand[k]))
    dy_rand = np.array(dy_rand)
    h_rand, _ = np.histogram(dy_rand, bins=dy_bins)
    mean_rand  = h_rand.mean()
    mc_excess_y.append(h_rand[pred_bin_y]/mean_rand - 1)
    mc_excess_z.append(h_rand[pred_bin_z]/mean_rand - 1)
    if (i+1) % 50 == 0: print(f'  MC {i+1}/{N_MC}')

mc_ey = np.array(mc_excess_y)
mc_ez = np.array(mc_excess_z)
sigma_y_node = (excess_y - mc_ey.mean()) / (mc_ey.std() + 1e-10)
sigma_z_node = (excess_z - mc_ez.mean()) / (mc_ez.std() + 1e-10)

print(f'\nSignificance at predicted spacings:')
print(f'  y-spacing {node_dy} Mpc: sigma = {sigma_y_node:.2f}')
print(f'  z-spacing {node_dz} Mpc: sigma = {sigma_z_node:.2f}')
print(f'  x=0 enhancement: {enhancement:.2f}x')

In [ ]:
# CELL 5 -- Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

# Panel 1: Node positions x vs y
ax = axes[0]
ax.scatter(nodes[:,0], nodes[:,1], s=3, alpha=0.5, color='#4FC3F7')
ax.axvline(0, color='yellow', lw=2, ls='--', label='x=0 plane (predicted)')
for n in range(-3, 4):
    ax.axhline(n*node_dy, color='orange', lw=0.8, ls=':', alpha=0.5)
ax.set_xlabel('x [Mpc]', color='white'); ax.set_ylabel('y [Mpc]', color='white')
ax.set_title('Node positions x-y\n(orange: predicted grid)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)

# Panel 2: y-separation histogram
ax = axes[1]
ax.bar(dy_centers, dy_hist, width=90, color='#888888', alpha=0.7)
ax.axvline(node_dy, color='#EF5350', lw=2.5, ls='--',
           label=f'T2/2={node_dy} Mpc (predicted)')
ax.axvline(node_dz, color='#42A5F5', lw=2.5, ls='--',
           label=f'T3/2={node_dz} Mpc (predicted)')
ax.set_xlabel('|dy| [Mpc]', color='white'); ax.set_ylabel('Count', color='white')
ax.set_title('Node y-separations\n(peaks at predicted spacings?)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)

# Panel 3: Significance
ax = axes[2]
labels = [f'y-spacing\n{node_dy} Mpc', f'z-spacing\n{node_dz} Mpc', f'x=0\nenhancement']
sigmas = [sigma_y_node, sigma_z_node, (enhancement-1)*5]  # scaled
colors = ['#EF5350', '#42A5F5', '#FFB74D']
bars = ax.bar(range(3), sigmas, color=colors, alpha=0.85, edgecolor='white')
for b, v in zip(bars, sigmas):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{v:.2f}', ha='center', color='white', fontsize=11, fontweight='bold')
ax.axhline(2,  color='orange', lw=1.5, ls='--', alpha=0.7, label='2sigma')
ax.axhline(-2, color='orange', lw=1.5, ls='--', alpha=0.7)
ax.set_xticks(range(3)); ax.set_xticklabels(labels, color='white', fontsize=9)
ax.set_title('P3 Significance\n(KEY RESULT)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

plt.suptitle('v46: Euclid P3 Node Density -- Cotting (2026)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('v46_euclid_p3.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# CELL 6 -- Summary
print('\n' + '='*60)
print('RESULT v46: EUCLID P3 NODE DENSITY')
print('Cotting, S. (2026) -- PRE-REGISTERED March 2026')
print('='*60)
print(f'  Nodes analysed: {len(nodes)}')
print(f'  Predicted spacing y: {node_dy} Mpc  sigma={sigma_y_node:.2f}')
print(f'  Predicted spacing z: {node_dz} Mpc  sigma={sigma_z_node:.2f}')
print(f'  x=0 enhancement: {enhancement:.2f}x')
print()
mean_sig = (sigma_y_node + sigma_z_node) / 2
if mean_sig > 2 and enhancement > 1.2:
    verdict = 'CONFIRMED -- Node grid at predicted spacings. KTF3 consistent.'
elif mean_sig > 1:
    verdict = 'MARGINAL -- Weak periodicity at predicted spacings.'
elif mean_sig < -1:
    verdict = 'FALSIFIED -- Anti-periodic at predicted spacings.'
else:
    verdict = 'NULL -- No node periodicity at predicted spacings.'
print(f'  Verdict: {verdict}')
print()
print('  NOTE: Only ~13 nodes expected in Euclid volume.')
print('  Statistical power is limited -- treat as indicative only.')
print('  Full Euclid mission (5 years) will have better statistics.')
print('='*60)